In [1]:
from util import import_ragas_custom

import_ragas_custom()

Arquivos copiados com sucesso!


In [2]:
import os

import pandas as pd

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from datasets import Dataset
from ragas.integrations.llama_index import evaluate
from ragas.testset.prompts import translate_prompts
from ragas.run_config import RunConfig
from ragas.metrics.critique import SUPPORTED_ASPECTS

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    load_index_from_storage,
)

from ragas.metrics import (
    answer_relevancy,
    answer_correctness,
    answer_similarity,
    context_precision,
    context_recall,
    context_utilization,
    context_entity_recall,
    noise_sensitivity_irrelevant,
    noise_sensitivity_relevant,
    faithfulness
)


In [3]:
DATA_PATH = 'data'
TESTSET = 'testset_llama3_2_3b.csv'
PERSIST_DIR = "./storage"
LANGUAGE = 'pt'
LANGUAGE_CACHE = 'cache'
TIMEOUT = 2400
RESULT_CSV = 'result_llama3_2_3b.csv'
MODEL = 'llama3.2'

metrics = [
    answer_relevancy,
    answer_correctness,
    answer_similarity,
    context_precision,
    context_recall,
    context_utilization,
    context_entity_recall,
    noise_sensitivity_irrelevant,
    noise_sensitivity_relevant,
    faithfulness
]

# metrics.extend(SUPPORTED_ASPECTS)


In [4]:
testset = pd.read_csv(TESTSET, index_col=0)
print("Tamanho do dataset: ", len(testset))
print(testset.head())

Tamanho do dataset:  114
                                            question  \
0  Qual é o papel da potência em um modelo de bat...   
1  Qual é o papel da tecnologia DSP no desenvolvi...   
2  Quais são as características principais da Tec...   
3  Qual benefício do Equalização Ativa do Banco d...   
4  Qual é o tipo de isolamento que é recomendado ...   

                                            contexts  \
0  ['       3,0          88 x 438 x 630          ...   
1  ['Alta Tecnologia em\nProcessamento Digital de...   
2  ['Perfection S erie Premium Estabilizador a IG...   
3  ['Equaliz erActive Batt ery Mangement\nBENEFÍC...   
4  ['Classe de Isolação:  \nF (150°C )\n Elev açã...   

                                        ground_truth evolution_type  \
0                                                NaN         simple   
1                                                NaN         simple   
2  A tecnologia DSP do Perfection S serie Premium...         simple   
3  O beneficio do

In [5]:
nan_rows = testset[testset.isna().any(axis=1)]
print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  28
                                              question  \
0    Qual é o papel da potência em um modelo de bat...   
1    Qual é o papel da tecnologia DSP no desenvolvi...   
7    Quais são as práticas recomendadas para manter...   
11   Quais são as características técnicas do Trans...   
13   Qual é o objetivo principal do sistema de sust...   
16   Qual é o papel do transformador de separação n...   
17   Qual é o papel da inovação e da qualidade na s...   
26   Qual é a importância do banco de dados em um s...   
31   Qual é o papel do processador de sinal digital...   
33   Qual é o papel da inovação em garantir a excel...   
36   Qual é o papel da tecnologia DSP na proteção d...   
38   Qual é o objetivo do No-Break Digital Signal P...   
40   Qual é o objetivo principal do sistema de reco...   
49   Qual é o papel das baterias nos sistemas de en...   
51           Qual o papel do estabilizador na formula?   
53   Qual vantagem é obtida com a arquitetura t

In [6]:
testset = testset.dropna()

In [7]:
print("Tamanho do dataset: ", len(testset))
print(testset.head())


Tamanho do dataset:  86
                                            question  \
2  Quais são as características principais da Tec...   
3  Qual benefício do Equalização Ativa do Banco d...   
4  Qual é o tipo de isolamento que é recomendado ...   
5  Qual é o papel do Gerenciamento de Baterias no...   
6  Quais são os benefícios da Equalização Ativa d...   

                                            contexts  \
2  ['Perfection S erie Premium Estabilizador a IG...   
3  ['Equaliz erActive Batt ery Mangement\nBENEFÍC...   
4  ['Classe de Isolação:  \nF (150°C )\n Elev açã...   
5  ['Equalização Ativa de BateriasEQUALIZER\nEqua...   
6  ['Equalização Ativa de BateriasEQUALIZER\nEqua...   

                                        ground_truth evolution_type  \
2  A tecnologia DSP do Perfection S serie Premium...         simple   
3  O beneficio do Equalizãó Ativa do Banco de Bat...         simple   
4                                          Até 250 V         simple   
5  O Gerenciamento

In [8]:
testset = Dataset.from_pandas(testset)

In [9]:
translate_prompts(LANGUAGE, LANGUAGE_CACHE)

In [10]:
embeding = OllamaEmbedding(model_name=MODEL)
model = Ollama(model=MODEL, request_timeout=TIMEOUT)

Settings.embed_model = embeding
Settings.llm = model

In [11]:
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader(DATA_PATH).load_data()
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

query_engine = index.as_query_engine(request_timeout=TIMEOUT)

In [12]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=model,
    embeddings=embeding,
    run_config=RunConfig(timeout=TIMEOUT, max_workers=2)
)

Running Query Engine:   0%|          | 0/86 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/860 [00:00<?, ?it/s]

Failed to parse output. Returning None.


-------------Prompt-------------


Dada a pergunta, resposta e contexto, verifique se o contexto foi útil para chegar à resposta dada. Dê o veredito como "1" se útil e "0" se não for com saída json.

A saída deve ser uma instância JSON bem formatada que esteja em conformidade com o esquema JSON abaixo.

Como exemplo, para o esquema {"motive": "string", "note": 0} o objeto {"motive": "Knife", "note": 7}.

Aqui está o esquema JSON de saída:
```
{"reason": "string", "verdict": 0}
```

Não retorne nenhum preâmbulo ou explicação, retorne apenas uma string JSON pura cercada por acentos graves triplos (```).

Examples:

question: "O que você pode me dizer sobre Albert Albert Einstein?"
context: "Albert Einstein (14 de março de 1879 – 18 de abril de 1955) foi um físico teórico nascido na Alemanha, amplamente considerado um dos maiores e mais influentes cientistas de todos os tempos. Mais conhecido por desenvolver a teoria da relatividade, ele também fe

In [13]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(RESULT_CSV)

In [14]:
print(result)

{'answer_relevancy': 0.6481, 'answer_correctness': 0.5240, 'answer_similarity': 0.6589, 'context_precision': 0.7534, 'context_recall': 0.4857, 'context_utilization': 0.6104, 'context_entity_recall': 0.0524, 'noise_sensitivity_irrelevant': 0.0909, 'noise_sensitivity_relevant': 0.0667, 'faithfulness': 0.5960}
